In [9]:
#Initializing Stuff
import sys
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import butter, filtfilt
import pandas as pad
import tkinter as tk
from tkinter import filedialog
import os
sys.path.append( '/Users/Dell/Documents/RESEARCH/tmsi-python-interface-main/tmsi-python-interface-main/TMSiFileFormats/file_readers')
from poly5reader import Poly5Reader



In [ ]:
#Get Data   
folder_path = "G:\\My Drive\\Research\\Data\\electrostim_data\\Doug 6-27\\20250627"
root =tk.Tk()
root.withdraw()  # Hide the root window
#select data (studying FLX only)
flexion_data = filedialog.askopenfilename(initialdir=folder_path, title="Select Flexion Data File")
#select baseline data (studying FLX only)
baseline_data = filedialog.askopenfilename(initialdir=folder_path, title="Select Baseline file")


In [11]:
accel_data_dict = {}

file_path = flexion_data
reader = Poly5Reader(file_path)
    #Extract Raw Samples (Channels x Samples)
samples = reader.samples
ch_names = reader.ch_names
sample_rate = reader.sample_rate
print(reader.ch_names)


trigger_channel = ['TRIGGERS']
trigger_indices = [ch_names.index(name) for name in trigger_channel if name in ch_names]
trigger_samples = samples [trigger_indices, :]
time_axis = np.arange(trigger_samples.shape[1])/ sample_rate


Reading file  G:/My Drive/Research/Data/electrostim_data/Doug 6-27/20250627/trial_7_FLX-20250627_152122.poly5
	 Number of samples:  123060 
	 Number of channels:  76 
	 Sample rate: 2000 Hz
Done reading data.
['CREF', 'R1C1', 'R1C2', 'R1C3', 'R1C4', 'R1C5', 'R1C6', 'R1C7', 'R1C8', 'R2C1', 'R2C2', 'R2C3', 'R2C4', 'R2C5', 'R2C6', 'R2C7', 'R2C8', 'R3C1', 'R3C2', 'R3C3', 'R3C4', 'R3C5', 'R3C6', 'R3C7', 'R3C8', 'R4C1', 'R4C2', 'R4C3', 'R4C4', 'R4C5', 'R4C6', 'R4C7', 'R4C8', 'R5C1', 'R5C2', 'R5C3', 'R5C4', 'R5C5', 'R5C6', 'R5C7', 'R5C8', 'R6C1', 'R6C2', 'R6C3', 'R6C4', 'R6C5', 'R6C6', 'R6C7', 'R6C8', 'R7C1', 'R7C2', 'R7C3', 'R7C4', 'R7C5', 'R7C6', 'R7C7', 'R7C8', 'R8C1', 'R8C2', 'R8C3', 'R8C4', 'R8C5', 'R8C6', 'R8C7', 'R8C8', 'BIP 01', 'BIP 02', 'BIP 03', 'BIP 04', 'ISO aux', 'ISO aux', 'AUX 1-3', 'AUX 2-1', 'TRIGGERS', 'STATUS', 'COUNTER']


In [ ]:
print(reader.ch_names[73])

In [ ]:
trigger = samples[73,:]
emg = samples[1:65]

grid_order = [
    16, 15, 14, 13, 12, 8, 4, 0,  # Row 1 (17,16,15,14,13, 9, 5, 1)
    21, 20, 19, 18, 17, 9, 5, 1,  # Row 2 (22…2)
    26, 25, 24, 23, 22, 10, 6, 2,  # Row 3
    31, 30, 29, 28, 27, 11, 7, 3,  # Row 4
    32, 33, 34, 35, 36, 52, 56, 60,  # Row 5 (33…61)
    37, 38, 39, 40, 41, 53, 57, 61,  # Row 6
    42, 43, 44, 45, 46, 54, 58, 62,  # Row 7
    47, 48, 49, 50, 51, 55, 59, 63   # Row 8
]

<![alt text](image.png)>

Indexed starting at 0, so channel 30 = channel 31
Channel 17 = 1-1 = R1C1

In [ ]:
%matplotlib ipympl
#RAW EMG CHANNELS BEFORE FILTERING

selected_channels = [0, 30, 32]  # Replace with the exact indices you want


# Number of selected channels
num_selected = len(selected_channels)


# Create subplots for each selected channel
fig, axes = plt.subplots(num_selected, 1, sharex=True, figsize=(20, 8))
plt.subplots_adjust(hspace=0.5)

# Loop over the selected channels and plot each one
for ax, ch_idx in zip(axes, selected_channels):
    ax.plot(time_axis, emg[ch_idx], label=f"Channel {ch_idx}")
    ax.set_ylabel("Amplitude (µV)")
    ax.legend(loc="upper right")

# Set common labels and title
axes[-1].set_xlabel("Time (s)")
fig.suptitle("Raw EMG Channels", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
#Smoothing/Bandpass Filtering
def bandpass_filter(x, lowcut, highcut, fs, order=4):
    nyq = fs/2
    b,a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, x)

def lpf_smooth(sig, cutoff, fs, order=5):
    nyq = fs/2
    b,a = butter(order, cutoff/nyq, btype='low')
    if sig.ndim==1:
        return filtfilt(b,a,sig)
    else:
        return np.vstack([filtfilt(b,a,chan) for chan in sig])

In [ ]:
fs          = 2000.0        # sampling rate
lowcut      = 30.0          # band-pass low end
highcut     = 400.0         # band-pass high end
bp_order    = 4

lpf_cutoff  = 20.0          # envelope smoothing cutoff
lpf_order   = 4

In [ ]:

# === 1) BAND-PASS raw EMG ===
# emg: (n_channels, n_samples)


selected_channels = [0, 30, 32] # Replace with the exact indices you want

filtered = np.zeros_like(emg)
for ch in range(emg.shape[0]):
    filtered[ch, :] = bandpass_filter(emg[ch, :],
                                       lowcut, highcut,
                                       fs, bp_order)

# === 2) LOW-PASS SMOOTH  *filtered* data ===
smoothed = lpf_smooth(filtered, lpf_cutoff, fs, lpf_order)

# === 3) PLOT the SMOOTHED EMG ===
fig, axes = plt.subplots(len(selected_channels), 1,
                         sharex=True, figsize=(12, 6))
plt.subplots_adjust(hspace=0.4)

for ax, ch in zip(axes, selected_channels):
    ax.plot(time_axis, smoothed[ch], color='C0', alpha=0.8)
    ax.set_ylabel(f"Ch {ch} (µV)")
    ax.set_title(f"Channel {ch} Smoothed EMG")

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Raw EMG → Band-pass → Low-pass Smoothing", fontsize=16)
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()

In [ ]:
# Plot the trigger channel
plt.figure(figsize=(15, 4))
plt.plot(time_axis, trigger, label="Trigger Channel (73)")
plt.xlabel("Time (s)")
plt.ylabel("Trigger Value")
plt.title("Trigger Channel (Index 73)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Find falling edges: trigger goes from 1 to 0
falling_edges = np.where(np.diff(trigger) < -0.5)[0] + 1  # +1 to get the index after the transition

print("Indices where trigger goes from 1 to 0:", falling_edges)

# To get the time between each trigger (in seconds):
falling_times = time_axis[falling_edges]
intervals = np.diff(falling_times)
print("Time between each trigger (s):", intervals)

In [ ]:

# Spike-Triggered Average (STA) using ±50 ms window 

# Parameters
window_ms = 50  # window before and after trigger in ms
window_samples = int(window_ms * sample_rate / 1000)  # samples per side
total_window = 2 * window_samples  # total window size (samples)

# Find trigger events (rising edges)
trigger_diff = np.diff(trigger)
trigger_onsets = np.where(trigger_diff > 0.5)[0] + 1  # adjust for diff offset

# Only keep triggers far enough from start/end for window
valid_triggers = trigger_onsets[
    (trigger_onsets > window_samples) & (trigger_onsets < trigger.shape[0] - window_samples)
]

# Extract EMG snippets around each trigger
snippets = []
for idx in valid_triggers:
    snippet = emg[:, idx - window_samples : idx + window_samples]
    snippets.append(snippet)
snippets = np.stack(snippets, axis=0)  # shape: (n_triggers, n_channels, window)

# Compute average across triggers
sta = np.mean(snippets, axis=0)  # shape: (n_channels, window)

# Time axis for window (centered at trigger)
sta_time = np.linspace(-window_ms, window_ms, total_window)

# Plot STA for selected channels
selected_channels = [0, 30, 32]
plt.figure(figsize=(12, 6))
for ch in selected_channels:
    plt.plot(sta_time, sta[ch], label=f'Ch {ch}')
plt.xlabel('Time (ms) relative to trigger')
plt.ylabel('EMG (µV)')
plt.title('Spike-Triggered Average (±50 ms window)')
plt.legend()
plt.tight_layout()
plt.show()